In [1]:

import re
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Create sample text corpus
corpus = """
The speed of transmission is an important point of difference between the two viruses. Influenza has a shorter median incubation period (the time from infection to appearance of symptoms) and a shorter serial interval (the time between successive cases) than COVID-19 virus. The serial interval for COVID-19 virus is estimated to be 5-6 days, while for influenza virus, the serial interval is 3 days. This means that influenza can spread faster than COVID-19.

Further, transmission in the first 3-5 days of illness, or potentially pre-symptomatic transmission –transmission of the virus before the appearance of symptoms – is a major driver of transmission for influenza. In contrast, while we are learning that there are people who can shed COVID-19 virus 24-48 hours prior to symptom onset, at present, this does not appear to be a major driver of transmission.

The reproductive number – the number of secondary infections generated from one infected individual – is understood to be between 2 and 2.5 for COVID-19 virus, higher than for influenza. However, estimates for both COVID-19 and influenza viruses are very context and time-specific, making direct comparisons more difficult.
"""

# Convert to lowercase, remove punctuation and split into words
corpus = corpus.lower()
corpus = re.sub(r'[^a-z0-9\s]', ' ', corpus)
tokens = corpus.split()

print("Total tokens:", len(tokens))

# 2. Create vocabulary and assign indexes
vocab = sorted(set(tokens))
vocab_size = len(vocab)

word_to_ix = {word: i for i, word in enumerate(vocab)}
ix_to_word = {i: word for i, word in enumerate(vocab)}

print("\nVocabulary:", vocab)
print("Vocabulary size:", vocab_size)

# 3. Generate context-target pairs
# Use two words on either side of the target word
window_size = 2
data = []

for i in range(window_size, len(tokens) - window_size):
    context = [
        tokens[i-2], tokens[i-1],
        tokens[i+1], tokens[i+2]
    ]
    target = tokens[i]
    data.append((context, target))

print("\nSample context-target pairs:")
for context, target in data[:5]:
    print(context, "->", target)

# Convert words into numerical indexes
X = torch.tensor(
    [[word_to_ix[w] for w in context] for context, target in data],
    dtype=torch.long
)

Y = torch.tensor(
    [word_to_ix[target] for context, target in data],
    dtype=torch.long
)

# 4. Build CBOW model using Embedding and Dense layers
torch.manual_seed(42)

embedding_dim = 20

embedding = nn.Embedding(vocab_size, embedding_dim)
fc = nn.Linear(embedding_dim, vocab_size)

loss_fn = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    list(embedding.parameters()) + list(fc.parameters()),
    lr=0.01
)

# 5. Train the model
epochs = 200

print("\nTraining...")

for epoch in range(epochs):

    optimizer.zero_grad()

    # Convert context words into embeddings
    embeds = embedding(X)

    # Average the context word embeddings
    avg_embeds = embeds.mean(dim=1)

    # Predict the target word
    output = fc(avg_embeds)

    loss = loss_fn(output, Y)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print("Epoch:", epoch + 1, "Loss:", loss.item())

# 6. Predict target word from given context
test_context = ["speed", "of", "is", "an"]

context = torch.tensor(
    [[word_to_ix[w] for w in test_context]],
    dtype=torch.long
)

with torch.no_grad():

    embeds = embedding(context)
    avg_embeds = embeds.mean(dim=1)
    output = fc(avg_embeds)

    # Convert scores into probabilities
    probabilities = torch.softmax(output, dim=1)

    predicted_idx = torch.argmax(probabilities, dim=1).item()
    predicted_word = ix_to_word[predicted_idx]

print("\nContext words:", test_context)
print("Predicted target word:", predicted_word)

# 7. Display learned word embeddings
print("\nLearned word embeddings:")

for word in ['influenza', 'covid', 'transmission', 'virus', 'days']:
    vec = embedding.weight[word_to_ix[word]].detach().numpy()
    print(word, ":", vec[:5])


Total tokens: 195

Vocabulary: ['19', '2', '24', '3', '48', '5', '6', 'a', 'an', 'and', 'appear', 'appearance', 'are', 'at', 'be', 'before', 'between', 'both', 'can', 'cases', 'comparisons', 'context', 'contrast', 'covid', 'days', 'difference', 'difficult', 'direct', 'does', 'driver', 'estimated', 'estimates', 'faster', 'first', 'for', 'from', 'further', 'generated', 'has', 'higher', 'hours', 'however', 'illness', 'important', 'in', 'incubation', 'individual', 'infected', 'infection', 'infections', 'influenza', 'interval', 'is', 'learning', 'major', 'making', 'means', 'median', 'more', 'not', 'number', 'of', 'one', 'onset', 'or', 'people', 'period', 'point', 'potentially', 'pre', 'present', 'prior', 'reproductive', 'secondary', 'serial', 'shed', 'shorter', 'specific', 'speed', 'spread', 'successive', 'symptom', 'symptomatic', 'symptoms', 'than', 'that', 'the', 'there', 'this', 'time', 'to', 'transmission', 'two', 'understood', 'very', 'virus', 'viruses', 'we', 'while', 'who']
Vocabular